# Notebook Instructions

1. If you are new to Jupyter notebooks, please go through this introductory manual <a href='https://quantra.quantinsti.com/quantra-notebook' target="_blank">here</a>.
1. Any changes made in this notebook would be lost after you close the browser window. **You can download the notebook to save your work on your PC.**
1. Before running this notebook on your local PC:<br>
i.  You need to set up a Python environment and the relevant packages on your local PC. To do so, go through the section on "**Run Codes Locally on Your Machine**" in the course.<br>
ii. You need to **download the zip file available in the last unit** of this course. The zip file contains the data files and/or Python modules that might be required to run this notebook.

# Reusing Access Token and Fetching Account Details

In the previous notebook you have learnt about the authentication and login process. This allows you to generate an access token, which stays valid for the entire day. Once you have this, you don’t need to repeat the login steps multiple times in a day.

In this notebook, you will:
1. Reuse that access token
2. Fetch basic account details
3. Fetch funds and margins data

## Import Libraries and Getting the API Key and Secret
As we did in the previous notebook, here we will first import the `kiteconnect` library and enter in the API key and API secret. 

But if we are reusing a stored `access_token`, why do we need to enter the API Key and API secret again? We need the API key to initialise `KiteConnect`. In addition to that, we need the API Secret to handle situations where the stored `access_token` might be invalid, and you would be required to perform a fresh login.

In [1]:
# Import the KiteConnect library
from kiteconnect import KiteConnect
# For data manipulation
import pandas as pd
# For sending logout request
import requests 
import os 

# --- CONFIGURATION ---
# It's highly recommended to use environment variables for your keys
# For this example, we'll assign them directly.
# In a real application, replace these with os.environ.get('YOUR_VARIABLE')
api_key = os.environ.get('ZERODHA_ALGOTRADER_API_KEY')
api_secret = os.environ.get('ZERODHA_ALGOTRADER_API_SECRET')

# Check if the keys are placeholders. If so, prompt the user.
if api_key == "YOUR_API_KEY" or api_secret == "YOUR_API_SECRET":
    print("ERROR: Please replace 'YOUR_API_KEY' and 'YOUR_API_SECRET' with your actual credentials.")
    # In a real script, you might exit here
    # exit()

# Initialize KiteConnect client
kite = KiteConnect(api_key=api_key)

print("KiteConnect client initialized.")

KiteConnect client initialized.


### Storing and Reusing the `access_token`

Here's a step by step of what you need to do:
1.  After generating the session, save the `access_token` to a local file (e.g., `access_token.txt`). The file must be saved in the same folder as this notebook. If this file is not in the same folder as your notebook, ensure the `token_path` in the code cell below is correct to prevent a `FileNotFoundError`.
2.  The next time your script runs, it first checks if this file exists and contains a token.
3.  If a token exists, it tries to use it. If the API call fails (e.g., the token has expired or it is invalid), it falls back to the manual login process.

> **Note:** The next cell reads the `access_token.txt` file.  
> Please make sure to run this notebook on your **local machine** where the file is available. Before you run the cells from this point onwards, you need to convert each cell type from "Raw" to "Code".

You will now use the stored `access_token` to fetch the profile details of your account by using the `kite.profile()` method. If a valid access_token is found, the output from the cell below will display your basic profile details.

In [3]:
#### Define a filename to store the token
token_path = "access_token.txt"

# --- Let's put it all together in a more practical script ---

# 1. Initialize KiteConnect
kite = KiteConnect(api_key=api_key)

access_token = None
# 2. Check if we already have a stored token from a previous session
try:
    with open(token_path, 'r') as f:
        access_token = f.read()
    kite.set_access_token(access_token)
    print("Found existing access token.")
except FileNotFoundError:
    print("Access token file not found. Proceeding with manual login.")
    pass # If file doesn't exist, we'll login manually

try:
    # 3. Validate the token by fetching profile information
    # If this call fails, it means the token is invalid or expired
    profile = kite.profile()
    print(f"\nWelcome, {profile['user_name']}! Authentication successful.")
    print(f"\nYour Account Details are as follows:")
    print(f"User ID: {profile['user_id']}")
    print(f"User Type: {profile['user_type']}")
    print(f"Email ID: {profile['email']}")
    print(f"User Short Name: {profile['user_shortname']}")
    print(f"Broker: {profile['broker']}")
    print(f"Exchanges: {profile['exchanges']}")
    print(f"Products: {profile['products']}")
    print(f"Order Types: {profile['order_types']}")
    
except Exception as e:
    print(f"Token validation failed: {e}. Starting fresh login flow.")

    # 4. If validation fails, start the manual login flow
    print(f"Please login using this URL:\n{kite.login_url()}")
    request_token = input("Enter the request_token from the redirect URL: ")

    # 5. Generate session and get the new access token
    try:
        data = kite.generate_session(request_token, api_secret=api_secret)
        access_token = data["access_token"]
        kite.set_access_token(access_token)

        # 6. Save the new token for future use
        with open(token_path, 'w') as f:
            f.write(access_token)
        print("\nNew session generated and token saved!")
        # Get profile for welcome message
        profile = kite.profile()
        print(f"\nWelcome, {profile['user_name']}! Authentication successful.")
        print(f"\nYour Account Details are as follows:")
        print(f"User ID: {profile['user_id']}")
        print(f"User Type: {profile['user_type']}")
        print(f"Email ID: {profile['email']}")
        print(f"User Short Name: {profile['user_shortname']}")
        print(f"Broker: {profile['broker']}")
        print(f"Exchanges: {profile['exchanges']}")
        print(f"Products: {profile['products']}")
        print(f"Order Types: {profile['order_types']}")

    except Exception as exc:
        print(f"Could not generate session: {exc}")
        exit() # Exit if we can't get a token

Found existing access token.

Welcome, Rhushabh Hasmukh Vaghela! Authentication successful.

Your Account Details are as follows:
User ID: IVN837
User Type: individual/res_no_nn
Email ID: rhushabhvaghela@gmail.com
User Short Name: Rhushabh
Broker: ZERODHA
Exchanges: ['NSE', 'MF']
Products: ['CNC', 'NRML', 'MIS', 'BO', 'CO']
Order Types: ['MARKET', 'LIMIT', 'SL', 'SL-M']


### Fetching Funds and Margins Data
In addition to your profile details, you can also fetch your funds and margin data. For this you will use the `kite.margins()` method.

In the code cell below we are fetching the funds and margins details associated with the `'equity'` segment. We will also transform this data into a Pandas DataFrame, making it easy to read in a tabular format. The `display()` function is used to show this table.

In [5]:
# Fetching funds and margins data
print("\nFetching funds and margins data...")
margins = kite.margins()

# Flatten the nested dictionary
equity = margins['equity']
flat_equity = {
    'enabled': equity['enabled'],
    'net': equity['net'],
}

# Add nested 'available' and 'utilised' fields with prefixes
for k, v in equity['available'].items():
    flat_equity[f'available_{k}'] = v

for k, v in equity['utilised'].items():
    flat_equity[f'utilised_{k}'] = v

# Create a DataFrame
equity_margins_df = pd.DataFrame([flat_equity])
display(equity_margins_df)


Fetching funds and margins data...


,enabled,net,available_adhoc_margin,available_cash,available_opening_balance,available_live_balance,available_collateral,available_intraday_payin,utilised_debits,utilised_payout,...,utilised_span,utilised_exposure,utilised_additional,utilised_delivery,utilised_option_premium,utilised_holding_sales,utilised_turnover,utilised_equity,utilised_m2m_realised,utilised_m2m_unrealised
0,True,0.5,0,0,-29.5,0.5,0,30,0,0,...,0,0,0,0,0,0,0,0,0,0


## Logging Out
Logging out invalidates the current `access_token` and ends the active API session. It ensures your access token can't be misused if someone gains access to your environment. If you want to switch accounts or reset your session, logging out is a safe way to do that. Explicitly closing your session avoids leaving unused sessions active on the Kite API server.

> Note: This does **not** log you out of the Kite web or mobile apps. It only ends the programmatic session used by your code.

#### What happens after logout?
After logging out, any further API calls will fail with an authentication error. The earlier stored `access_token` will now be invalidated. You would need to go through the login process again and generate a new `access_token`.

In [6]:
# Construct the logout URL using query parameters
url = f"https://api.kite.trade/session/token?api_key={api_key}&access_token={access_token}"

# Set the required API version header
headers = {
    "X-Kite-Version": "3"
}

# Make a DELETE request to invalidate the session
response = requests.delete(url, headers=headers)

# Check the response
if response.status_code == 200:
    # Successful logout
    print("Logout successful:", response.json())
else:
    # Logout failed — print the error
    print("Logout failed:", response.text)

Logout successful: {'status': 'success', 'data': True}


## Conclusion

You have now successfully navigated the KiteConnect authentication process. While it involves a manual step, the ability to save and reuse the `access_token` makes it manageable for daily operations. With the authenticated `kite` object, you are now ready to explore the KiteConnect API to fetch data, manage orders, and build your trading application.
<br><br>